## ML-10 - Bronze ingestion | validation

Investigação técnica para validar a abordaem de ingestão Bronze antes da promoção para `notebooks/01_bronze/ingest_raw`

Objetivos:

- validar a construção de `AppConfig` a partir de parâmetros explícitos de execução
- reutilizar os contratos Bronze já definidos em `_shared/contracts.py`
- ler os CSVs raw usando schema explícito, sem `inferSchema`
- preservar os campos semiestruturados como `STRING`
- adicionar `_source_file`, `_ingested_at` e `_ingestion_id`
- validar escrita Delta no namespace Bronze
- registrar evidências e observações que justifiquem a implementação produtiva.


## Step 1. Imports

In [0]:
from datetime import datetime, timezone
from uuid import uuid4

from pyspark.sql import functions as F
from pyspark.sql.types import StructType

from notebooks._shared.configuration import AppConfig
from notebooks._shared.contracts import BRONZE_MOVIES, BRONZE_CREDITS

## Step 2. Functions

In [0]:
def bronze_schema_without_metadata(contract):
    metadata_columns = {
        "_source_file",
        "_ingested_at",
        "_ingestion_id",
    }

    return StructType(
        [
            field
            for field in contract.schema.fields
            if field.name not in metadata_columns
        ]
    )


def validate_persisted_schema_against_contract(df, contract):
    actual_schema = df.schema
    expected_schema = contract.schema

    if actual_schema != expected_schema:
        raise ValueError(
            f"Schema persistido divergente para {contract.name}.\n"
            f"Actual:   {actual_schema.simpleString()}\n"
            f"Expected: {expected_schema.simpleString()}"
        )

    print(
        f"{contract.name}: schema persistido aderente ao contrato "
        f"({len(actual_schema.fields)} campos)"
    )

def validate_schema_against_contract(df, contract):
    actual_fields = df.schema.fields
    expected_fields = contract.schema.fields

    if len(actual_fields) != len(expected_fields):
        raise ValueError(
            f"Qtd de campos divergentes para {contract.name}: "
            f"actual={len(actual_fields)}, expected={len(expected_fields)}"
        )

    differences = []

    for position, (actual, expected) in enumerate(
        zip(actual_fields, expected_fields), start=1
    ):
        if actual.name != expected.name:
            differences.append(
                f"posição {position}: "
                f"name actual={actual.name}, expected={expected.name}"
            )

        if actual.dataType != expected.dataType:
            differences.append(
                f"posição {position} ({expected.name}): "
                f"type actual={actual.dataType.simpleString()}, "
                f"expected={expected.dataType.simpleString()}"
            )
            
    if differences:
        raise ValueError(
            f"Schema divergente para {contract.name}:\n" + "\n".join(differences)
        )

    print(f"{contract.name}: schema aderente ao contrato ({len(actual_fields)} campos)")

## Step 3 - Paramêtros de execução

In [0]:
dbutils.widgets.text("catalog","movielakehouse")
dbutils.widgets.text("bronze_schema","bronze")
dbutils.widgets.text("raw_volume","raw")

## Step 4. Configuração

In [0]:
config = AppConfig(
    catalog=dbutils.widgets.get("catalog"),
    bronze_schema=dbutils.widgets.get("bronze_schema"),
    raw_volume=dbutils.widgets.get("raw_volume"),
)

movies_source_path = f"{config.raw_volume_path}/tmdb_5000_movies.csv"
credits_source_path = f"{config.raw_volume_path}/tmdb_5000_credits.csv"

movies_source_schema = bronze_schema_without_metadata(BRONZE_MOVIES)
credits_source_schema = bronze_schema_without_metadata(BRONZE_CREDITS)

print(config.as_dict())
print(f"Movies source: {movies_source_path}")
print(f"Credits source: {credits_source_path}")

## Step 5. Schemas
Derivar a partir dos contratos Bronze completos, os schemas utilizados na leitura dos arquivos CSV, excluindo apenas
os metadatos técnicos que serão add durante a ingestão

In [0]:

print("Movies source schema:")
print(movies_source_schema.simpleString())

print("\nCredits source schema:")
print(credits_source_schema.simpleString())

In [0]:
print(
    "\nMovies:",
    f"source={len(movies_source_schema.fields)}",
    f"| bronze_schema={len(BRONZE_MOVIES.schema.fields)}"
)

print(
    "\nCredits:",
    f"source={len(credits_source_schema.fields)}",
    f"| bronze_schema={len(BRONZE_CREDITS.schema.fields)}"
)

## Step 6. Inspeção dos contratos
Inspectionar os contrator  Bronze compeltos e confirmar que eles representam as colunas a fonte
acrescidas dos metadados técnicos da ingestão

In [0]:
print("BRONZE_MOVIES")
print(BRONZE_MOVIES.schema.simpleString())
print(f"Field count: {len(BRONZE_MOVIES.schema.fields)}")

print()

print("BRONZE_CREDITS")
print(BRONZE_CREDITS.schema.simpleString())
print(f"Field count: {len(BRONZE_CREDITS.schema.fields)}")

## Step 7. Validação dos headers da fonte
Validar que os headers físicos dos arquivos CSV correspondem às colunas esperadas pelos contratos Bronze antes da leitura contratual e da persistência

In [0]:
movies_header = (
    spark.read.option("header", True)
    .option("multiline", True)
    .option("quote", '"')
    .option("escape", '"')
    .csv(movies_source_path)
    .columns
)

credits_header = (
    spark.read.option("header", True)
    .option("multiline", True)
    .option("quote", '"')
    .option("escape", '"')
    .csv(credits_source_path)
    .columns
)

expected_movies_header = [field.name for field in movies_source_schema.fields]

expected_credits_header = [field.name for field in credits_source_schema.fields]

assert movies_header == expected_movies_header, (
    f"Movies header divergente.\n"
    f"Actual:   {movies_header}\n"
    f"Expected: {expected_movies_header}"
)

assert credits_header == expected_credits_header, (
    f"Credits header divergente.\n"
    f"Actual:   {credits_header}\n"
    f"Expected: {expected_credits_header}"
)

print(f"Movies: header aderente ({len(movies_header)} colunas)")
print(f"Credits: header aderente ({len(credits_header)} colunas)")

## Step 8. Leitura dos CSVs
Ler os dados de origem com schema sxplcítio dericado dos contratos da bronze, preservando os valore da fonte como STRING

In [0]:
movies_df = (
    spark.read
    .option("header", True)
    .option("multiline", True)
    .option("quote", '"')
    .option("escape", '"')
    .schema(movies_source_schema)
    .csv(movies_source_path)
)

credits_df = (
    spark.read
    .option("header", True)
    .option("multiline", True)
    .option("quote", '"')
    .option("escape", '"')
    .schema(credits_source_schema)
    .csv(credits_source_path)
)

print(f"Movies Path: {movies_source_path}")
print(f"Movies rows: {movies_df.count()}")
movies_df.printSchema()

print()

print(f"Credits Path: {credits_source_path}")
print(f"Credits rows: {credits_df.count()}")
credits_df.printSchema()

## Step 9. Metadados
Add aos DFs de origem os metadados técnicos definidos no contrato Bronze

In [0]:
ingestion_id = str(uuid4())
ingested_at = datetime.now(timezone.utc)

movies_bronze_df = (
    movies_df
    .withColumn("_source_file", F.lit(movies_source_path))
    .withColumn("_ingested_at", F.lit(ingested_at))
    .withColumn("_ingestion_id", F.lit(ingestion_id))
)

credits_bronze_df = (
    credits_df
    .withColumn("_source_file", F.lit(credits_source_path))
    .withColumn("_ingested_at", F.lit(ingested_at))
    .withColumn("_ingestion_id", F.lit(ingestion_id))
)

print(f"IngestionID: {ingestion_id}")
print(f"IngestedAt: {ingested_at}")
print()

movies_bronze_df.select(
    "_source_file",
    "_ingested_at",
    "_ingestion_id",
).show(2, truncate=False)

credits_bronze_df.select(
    "_source_file",
    "_ingested_at",
    "_ingestion_id",
).show(2, truncate=False)

## Step 10. Validação do contrato
Validar que os DFs Bronze preparaos para a persistência aderem aos contratos estructurais definidos para Movies e Credits,
comparando qyd, nomes, ordem e tipos da colunas

In [0]:
validate_schema_against_contract(movies_bronze_df, BRONZE_MOVIES)

validate_schema_against_contract(credits_bronze_df, BRONZE_CREDITS)

## Step 11. Escrita na Delta
Persistir os DFs preparados como tabelas Delta gerenciados no schema bronze, utilizando os nomes definidos no respectivo contrato

In [0]:
movies_table = f"{config.bronze_namespace}.{BRONZE_MOVIES.name}"
credits_table = f"{config.bronze_namespace}.{BRONZE_CREDITS.name}"

print(f"Movies table: {movies_table}")
print(f"Credits table: {credits_table}")

In [0]:
(
    movies_bronze_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(movies_table)
)

(
    credits_bronze_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(credits_table)
)

print("Encrita em bronze concluída")

# Usamos overwrite pq ainda estamos no EDA da ML-10 validando técnicamente a ingestão, sem transformar a descição de indempotência

In [0]:
# %sql
# DROP TABLE IF EXISTS movielakehouse.bronze.movies;
# DROP TABLE IF EXISTS movielakehouse.bronze.credits;

In [0]:
# spark.sql(f"DROP TABLE IF EXISTS {movies_table}")
# spark.sql(f"DROP TABLE IF EXISTS {credits_table}")

# print("Tabelas experimentais Bronze removidas.")

## Step 12. validação das tbls Bronze persistidas
Reler as tbls Bronze a partir do UC e validar que a persistência foi concluída com as qtd esperadas e com a estrutura compatível com os contratos definidos

In [0]:
movies_table_df = spark.table(movies_table)
credits_table_df = spark.table(credits_table)

print(f"Movies rows:  {movies_table_df.count()}")
print(f"Credits rows: {credits_table_df.count()}")

print()
print("Esquema persistente: Movies ")
movies_table_df.printSchema()

print()
print("Esquema persistente: Credits ")
credits_table_df.printSchema()

## Step 13. Comparação do schema persistido com os ontatos Bronze
Comprar os schema relidos das tbls Delta com os contratos Bronze completo, validando nomes, ordem, tipos e nulabilidade após a persistência

In [0]:
validate_persisted_schema_against_contract(
    movies_table_df,
    BRONZE_MOVIES,
)

validate_persisted_schema_against_contract(
    credits_table_df,
    BRONZE_CREDITS,
)

## Step 14. Preservação da identidade da fonte

confirmar que a ingestão Bronze preservou as características de identidade e correspondência observadas na fonte, sem antecipar as validações formais de qualidade.

In [0]:
movies_null_ids = movies_table_df.filter(F.col("id").isNull()).count()
movies_distinct_ids = movies_table_df.select("id").distinct().count()

credits_null_ids = credits_table_df.filter(F.col("movie_id").isNull()).count()
credits_distinct_ids = credits_table_df.select("movie_id").distinct().count()

movies_without_credits = (
    movies_table_df.select(F.col("id").alias("movie_id"))
    .join(
        credits_table_df.select("movie_id"),
        on="movie_id",
        how="left_anti",
    )
    .count()
)

credits_without_movies = (
    credits_table_df.select("movie_id")
    .join(
        movies_table_df.select(F.col("id").alias("movie_id")),
        on="movie_id",
        how="left_anti",
    )
    .count()
)

print(f"Movies null ids: {movies_null_ids}")
print(f"Movies distinct ids: {movies_distinct_ids}")
print(f"Credits null ids: {credits_null_ids}")
print(f"Credits distinct ids: {credits_distinct_ids}")
print(f"Movies without credits: {movies_without_credits}")
print(f"Credits without movies: {credits_without_movies}")